## 1. Import Required Libraries

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import os
import time
import itertools
from numpy import dot, mean, zeros, sqrt, zeros_like, tile, sum
from numpy.linalg import det
from sklearn.datasets import make_regression
from joblib import Parallel, delayed

matplotlib.use('Agg')

## 2. SGD Optimizer

In [1]:
class SGD:
    def __init__(self, learning_rate):
        self.learning_rate = learning_rate

    def step(self, weights, bias, dw, db):
        # Update weights and bias using gradient descent
        weights -= self.learning_rate * dw
        bias -= self.learning_rate * db
        return weights, bias

## 3. ADAM Optimizer

In [12]:
class ADAM:
    def __init__(self, learning_rate, b1=0.9, b2=0.999):
        self.learning_rate, self.b1, self.b2 = learning_rate, b1, b2
        self.mw, self.vw = None, None
        self.mb, self.vb = None, None
        self.t = 0
        self.epsilon = 1e-8

    def step(self, weights, bias, dw, db):
        # Initialize moment vectors
        if self.mw is None:
            self.mw, self.vw = zeros_like(weights), zeros_like(weights)
            self.mb, self.vb = zeros_like(bias), zeros_like(bias)

        self.t += 1
        # Update biased first and second moment estimates
        self.mw = self.b1 * self.mw + (1 - self.b1) * dw
        self.vw = self.b2 * self.vw + (1 - self.b2) * dw**2
        self.mb = self.b1 * self.mb + (1 - self.b1) * db
        self.vb = self.b2 * self.vb + (1 - self.b2) * db**2

        # Compute bias-corrected estimates
        mw_h = self.mw / (1 - self.b1**self.t)
        vw_h = self.vw / (1 - self.b2**self.t)
        mb_h = self.mb / (1 - self.b1**self.t)
        vb_h = self.vb / (1 - self.b2**self.t)

        # Update parameters
        weights -= (self.learning_rate / (sqrt(vw_h) + self.epsilon)) * mw_h
        bias -= (self.learning_rate / (sqrt(vb_h) + self.epsilon)) * mb_h
        return weights, bias

## 4. DREM Optimizer

In [13]:
class DREM:
    def __init__(self, alpha=0.01):
        self.alpha = alpha

    def step(self, weights, x_b, y_b):
        delta = det(x_b)
        
        # Stability check
        if abs(delta) <= 1 or abs(delta) >= 50:
            return weights

        size = x_b.shape[0]
        Y_vector = zeros((size, 1))

        # Solve for Y_vector using Cramer's rule-like approach
        for i in range(size):
            Xi = x_b.copy()
            Xi[:, i] = y_b.flatten()
            Y_vector[i] = det(Xi)
        
        # Weight update based on determinant-driven rule can be standart or adapt (then uncomment the denominator)
        weights -= (self.alpha * delta * (delta * weights - Y_vector))# / (1 + self.alpha * (delta ** 2))
            
        return weights

## 5. Perceptron Model

In [14]:
class Perseptron:
    def __init__(self, weights, bias, n_neurons):
        self.weights = weights
        self.bias = bias
        self.n_neurons = n_neurons

    def forward(self, x):
        return mean(dot(x, self.weights) + self.bias, axis=1, keepdims=True)

    def MSE(self, y, y_pred):
        return mean((y - y_pred)**2)

    def backward(self, x, y, y_p, total_n):
        # Calculate gradients for weights and bias
        grad_common = -(2 / (x.shape[0] * self.n_neurons)) * (y - y_p)
        
        dw = dot(x.T, grad_common)
        db = sum(grad_common, axis=0, keepdims=True)
        return tile(dw, (1, self.n_neurons)), tile(db, (1, self.n_neurons))

## 6. Training Function

In [15]:
def train(X_ext, y, n_neurons, optimizer_type, epochs=20):
    n_samples, n_features_ext = X_ext.shape
    np.random.seed(42)
    w_full = np.random.uniform(0, 1, (n_features_ext, n_neurons))

    # Initialize based on optimizer type
    if isinstance(optimizer_type, DREM):
        weights = w_full.copy()
        model = None
    else:
        weights = w_full[1:].copy()
        bias = w_full[0:1].copy()
        model = Perseptron(weights, bias, n_neurons)

    loss_history = []
   
    for epoch in range(epochs):
        # Forward pass
        if isinstance(optimizer_type, DREM):
            pred = mean(dot(X_ext, weights), axis=1, keepdims=True)
        else:
            pred = model.forward(X_ext[:, 1:])

        loss_history.append(mean((y - pred)**2))

        # Shuffle data
        idx = np.random.permutation(n_samples)
        xs, ys = X_ext[idx], y[idx]
        bs = n_features_ext if isinstance(optimizer_type, DREM) else 5

        # Mini-batch loop
        for i in range(0, n_samples, bs):
            xb, yb = xs[i:i+bs], ys[i:i+bs]
            if xb.shape[0] < bs: continue

            if isinstance(optimizer_type, DREM):
                weights = optimizer_type.step(weights, xb, yb)
            else:
                y_p = model.forward(xb[:, 1:])
                dw, db = model.backward(xb[:, 1:], yb, y_p, n_samples)
                model.weights, model.bias = optimizer_type.step(model.weights, model.bias, dw, db)   
    return loss_history


## 7. Experiment Function

In [16]:
def run_experiment(args):
    try:
        # Determine the source of the data based on the arguments length
        if len(args) == 5:
            i, combo, folder_name, X_raw, y_raw = args
            is_synthetic = False
        else:
            i, combo, folder_name = args
            # Generate synthetic data if external data is not provided
            X_raw, y_flat = make_regression(
                n_samples=combo.get('n_samples', 100),
                n_features=combo.get('n_features', 10),
                noise=combo.get('noise', 0.1),
                random_state=42
            )
            y_raw = y_flat.reshape(-1, 1)
            is_synthetic = True

        if X_raw is None or len(X_raw) == 0:
            return None

        # Data normalization
        X_mean = X_raw.mean(axis=0)
        X_std = X_raw.std(axis=0)
        X = (X_raw - X_mean) / (X_std + 1e-8)
        
        y_mean = y_raw.mean(axis=0)
        y_std = y_raw.std(axis=0)
        y = (y_raw - y_mean) / (y_std + 1e-8)
        
        # Add bias column
        X_ext = np.hstack([np.ones((X.shape[0], 1)), X])

        iteration_results = []
        histories_drem, histories_adam, histories_sgd = [], [], []

        # Run multiple training cycles for averaging
        for j in range(3):

            # SGD
            start_sgd = time.perf_counter()
            loss_sgd = train(X_ext, y, n_neurons=combo['n_neurons'], 
                             optimizer_type=SGD(learning_rate=combo['alpha']), epochs=combo['epochs'])
            time_sgd = time.perf_counter() - start_sgd
            
            # Adam
            start_adam = time.perf_counter()
            loss_adam = train(X_ext, y, n_neurons=combo['n_neurons'], 
                              optimizer_type=ADAM(learning_rate=combo['alpha']), epochs=combo['epochs'])
            time_adam = time.perf_counter() - start_adam
            
            # DREM
            start_drem = time.perf_counter()
            loss_drem = train(X_ext, y, n_neurons=combo['n_neurons'], 
                              optimizer_type=DREM(alpha=combo['alpha']), epochs=combo['epochs'])
            time_drem = time.perf_counter() - start_drem

            iteration_results.append({
                'mse_drem': loss_drem[-1], 
                'mse_adam': loss_adam[-1],
                'mse_sgd': loss_sgd[-1], 
                'time_drem': time_drem,
                'time_adam': time_adam,
                'time_sgd': time_sgd
            })
            histories_drem.append(loss_drem)
            histories_adam.append(loss_adam)
            histories_sgd.append(loss_sgd)

        # Visualization
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(np.mean(histories_drem, axis=0), label='DREM', color='red')
        ax.plot(np.mean(histories_adam, axis=0), label='Adam', color='blue')
        ax.plot(np.mean(histories_sgd, axis=0), label='SGD', color='green')
        
        # Dynamic title and filename generation based on data type
        if is_synthetic:
            title_str = (f"Synthetic: S={combo.get('n_samples')} F={combo.get('n_features')} N={combo.get('noise')}\n"
                         f"Neu={combo['n_neurons']} Alpha={combo['alpha']} Ep={combo['epochs']}")
            file_name = f"exp_{i}_s{combo.get('n_samples')}_f{combo.get('n_features')}_n{combo['n_neurons']}.png"
        else:
            title_str = (f"Real Data: Samples={X_raw.shape[0]} Features={X_raw.shape[1]}\n"
                         f"Neu={combo['n_neurons']} Alpha={combo['alpha']} Ep={combo['epochs']}")
            file_name = f"exp_{i}_neu{combo['n_neurons']}_a{combo['alpha']}_ep{combo['epochs']}.png"

        ax.set_title(title_str)
        ax.legend()
        ax.grid(True)
        
        fig.savefig(os.path.join(folder_name, file_name))
        plt.close(fig)

        return {
            'num': i,
            'mse_drem': np.mean([x['mse_drem'] for x in iteration_results]),
            'mse_adam': np.mean([x['mse_adam'] for x in iteration_results]),
            'mse_sgd': np.mean([x['mse_sgd'] for x in iteration_results]),
            'time_drem': np.mean([x['time_drem'] for x in iteration_results]),
            'time_adam': np.mean([x['time_adam'] for x in iteration_results]),
            'time_sgd': np.mean([x['time_sgd'] for x in iteration_results]),
            **combo
        }
    except Exception as e:
        print(f"Error in task {i}: {e}")
        return None

## 8. Run Experiments: make_regression

In [ ]:
if __name__ == '__main__':
    params_grid = {
    'n_neurons': [1, 5, 10, 25],
    'alpha': [0.0001, 0.0005, 0.001, 0.005],
    'epochs': [5, 8, 15, 20, 30],
    'n_samples': [100, 1000, 10000],
    'n_features': [5, 10, 20],
    'noise': [1.0, 5.0, 10.0]
    }

    folder_name = 'experiment_results_mr_standart'
    if not os.path.exists(folder_name): 
        os.makedirs(folder_name)

    # Generate all possible hyperparameter combinations (Grid Search)
    keys, values = zip(*params_grid.items())
    combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
    
    # Prepare the list of tasks for parallel processing
    tasks = [(i, c, folder_name) for i, c in enumerate(combinations, 1)]
    total_tasks = len(tasks)
    
    # Execute experiments in parallel using 13 CPU cores
    results_list = Parallel(n_jobs=13, verbose=10)(
        delayed(run_experiment)(task) for task in tasks
    )

    results_list = [res for res in results_list if res is not None]

    if results_list:
        results_list.sort(key=lambda x: x['num'])
        pd.DataFrame(results_list).to_csv("experiment_results_mr_standart.csv", index=False)
        print("Results saved to experiment_results_mr_standart.csv")
    else:
        print("File was not created.")

Starting 2160 tasks on 13 cores...


[Parallel(n_jobs=13)]: Using backend LokyBackend with 13 concurrent workers.
[Parallel(n_jobs=13)]: Done   6 tasks      | elapsed:    0.9s
[Parallel(n_jobs=13)]: Done  15 tasks      | elapsed:    1.3s
[Parallel(n_jobs=13)]: Done  24 tasks      | elapsed:    1.8s
[Parallel(n_jobs=13)]: Done  35 tasks      | elapsed:    3.7s
[Parallel(n_jobs=13)]: Done  46 tasks      | elapsed:    6.6s
[Parallel(n_jobs=13)]: Done  59 tasks      | elapsed:    9.9s


## 9. Run Experiments: AutoMPG Dataset

In [ ]:
# MPG experiment results:
#kaggle link: https://www.kaggle.com/datasets/yasserh/auto-mpg-dataset

if __name__ == '__main__':
    folder_name = 'experiment_results_autompg_standart'
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)

    df = pd.read_csv('auto-mpg.csv')
    
    # Preprocessing: replace missing values, drop rows with NaNs, and remove duplicates
    df = df.replace('?', np.nan)
    df = df.dropna().drop_duplicates()
    
    # Remove the 'car name' column as it is non-numeric and not suitable for regression
    if 'car name' in df.columns:
        df = df.drop(columns=['car name'])
    
    df = df.astype(np.float64)
        
    # Separate into feature matrix X and target variable y (mpg)
    y_col = 'mpg'
    X_df = df.drop(columns=[y_col])
    
    X_data = X_df.values
    y_data = df[y_col].values.reshape(-1, 1)

    params_grid = {
        'n_neurons': [1, 5, 10, 25],
        'alpha': [0.0001, 0.0005, 0.001, 0.005],
        'epochs': [5, 8, 15, 20, 30]
    }

    keys, values = zip(*params_grid.items())
    combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
    
    tasks = [(i, c, folder_name, X_data, y_data) for i, c in enumerate(combinations, 1)]
    total_tasks = len(tasks)
    
    results_list = Parallel(n_jobs=13)(delayed(run_experiment)(task) for task in tasks)

    if results_list:
        results_list.sort(key=lambda x: x['num'])
        pd.DataFrame(results_list).to_csv("research_results_autompg_standart.csv", index=False)
        print("Results saved to research_results_autompg_standart.csv")

## 10. Run Experiments: Salary Dataset

In [ ]:
#Salary_data experiment results
#kaggle link: https://www.kaggle.com/datasets/wardabilal/salary-prediction-dataset

from sklearn.preprocessing import LabelEncoder

if __name__ == '__main__':
    folder_name = 'experiment_results_salary_standart'
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)

    df = pd.read_csv('Salary_Data.csv')
    df = df.dropna().drop_duplicates()  # Remove missing values and duplicates

    if 'Age' in df.columns:
        df = df.drop(columns=['Age'])

    categorical_cols = ['Gender', 'Education Level', 'Job Title']
    le = LabelEncoder()
    for col in categorical_cols:
        df[col] = le.fit_transform(df[col])

    y_col = 'Salary'
    X_df = df.drop(columns=[y_col])
    
    X_data = X_df.values.astype(np.float64)
    y_data = df[y_col].values.astype(np.float64).reshape(-1, 1)

    params_grid = {
        'n_neurons': [1, 5, 10, 25],
        'alpha': [0.0001, 0.0005, 0.001, 0.005],
        'epochs': [5, 8, 15, 20, 30]
    }

    keys, values = zip(*params_grid.items())
    combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

    tasks = [(i, c, folder_name, X_data, y_data) for i, c in enumerate(combinations, 1)]
    total_tasks = len(tasks)

    # Execute experiments in parallel using 13 CPU cores
    results_list = Parallel(n_jobs=13)(delayed(run_experiment)(task) for task in tasks)

    if results_list:
        results_list.sort(key=lambda x: x['num'])
        pd.DataFrame(results_list).to_csv("research_results_salary_standart.csv", index=False)
        print("Results saved to research_results_salary_standart.csv")